# Colab TTS Quality Gate

Run this notebook on Google Colab with a T4 (GPU runtime).
Do NOT run this locally. The notebook installs only the TTS extras from requirements-tts.txt and a CUDA-enabled torch wheel.

Outputs:
- projects/colab-tts-quality/qwen3_tts_output.wav
- projects/colab-tts-quality/chatterbox_output.wav
- projects/colab-tts-quality/tts_quality_report.json


In [ ]:
# Check GPU availability and basic environment
import torch, subprocess, os, time
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    try:
        print('GPU name:', torch.cuda.get_device_name(0))
    except Exception as e:
        print('Could not query device name:', e)
    print('Device count:', torch.cuda.device_count())
    try:
        import pynvml
        pynvml.nvmlInit()
        h = pynvml.nvmlDeviceGetHandleByIndex(0)
        mem = pynvml.nvmlDeviceGetMemoryInfo(h)
        print('VRAM total (MB):', mem.total // 1024 // 1024)
    except Exception as e:
        print('pynvml not available (will attempt torch memory info).', e)
else:
    print('No CUDA GPU detected ? stop and switch to Colab GPU runtime.')


In [ ]:
# Install CUDA torch first (adjust cu118 to your Colab CUDA if needed).
# Then install only the TTS extras from requirements-tts.txt.
# This cell is intentionally explicit and must be run in Colab.
import sys, os
print('Installing dependencies ? this may take a few minutes')
!python -m pip install --upgrade pip setuptools wheel -q
# Install CUDA-enabled torch for Colab T4 (cu118 often works). Adjust if necessary.
!python -m pip install --index-url https://download.pytorch.org/whl/cu118 --upgrade torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu118 -q
# Install the TTS extras from the repository file (assumes requirements-tts.txt is present in the repo root)
!python -m pip install -r requirements-tts.txt -q
# Install ffmpeg for validation if not present
!apt-get update -qq && apt-get install -y -qq ffmpeg
print('Install complete')


In [ ]:
# Prepare workspace
from pathlib import Path, PurePosixPath
out_dir = Path('projects/colab-tts-quality')
out_dir.mkdir(parents=True, exist_ok=True)
qwen_path = out_dir / 'qwen3_tts_output.wav'
chatter_path = out_dir / 'chatterbox_output.wav'
report_path = out_dir / 'tts_quality_report.json'
text = (
    'A hundred years ago, humanity looked toward the stars and wondered whether we were alone.\n\n'
    'Tonight, something answered.\n\n'
    'The signal came from a world no telescope had ever seen before.\n\n'
    'And buried inside that transmission was a message meant for us.'
)
instruction = (
    'Calm cinematic documentary narration. Natural pacing. Clear pronunciation. Slight sense of mystery and anticipation. Subtle emotional expression. Avoid sounding like an advertisement or an AI assistant.'
)
print('Text length:', len(text))


In [ ]:
# Helper utilities: ffprobe validation, wav info, timings, VRAM sampling
import json, subprocess, time, os
import torch
def ffprobe_info(path):
    cmd = ['ffprobe','-v','error','-show_format','-show_streams','-print_format','json', str(path)]
    p = subprocess.run(cmd, capture_output=True, text=True)
    if p.returncode != 0:
        return None, p.stderr
    return json.loads(p.stdout), None
def file_ok(path):
    if not os.path.exists(path):
        return False, 'missing'
    size = os.path.getsize(path)
    if size == 0:
        return False, 'zero-size'
    info, err = ffprobe_info(path)
    if info is None:
        return False, f'ffprobe-failed: {err}'
    return True, info
def vram_snapshot():
    if not torch.cuda.is_available():
        return {}
    torch.cuda.synchronize()
    return {
        'allocated': torch.cuda.memory_allocated(0),
        'reserved': torch.cuda.memory_reserved(0),
        'max_allocated': torch.cuda.max_memory_allocated(0),
        'max_reserved': torch.cuda.max_memory_reserved(0),
    }


## Qwen3-TTS test (0.6B) ? REAL only
If this fails to import or instantiate the real model, the test will report `QWEN3 REAL = FAIL` and show the exception.


In [ ]:
qwen_result = { 'model': 'Qwen3-TTS-0.6B' }
try:
    import time, os
    start_load = time.time()
    # Import the official qwen3tts interface as used by adapter code
    from qwen3tts.utils import TTS as Qwen3TTS
    import torch
    load_time = time.time() - start_load
    qwen_result['load_time'] = load_time
    # VRAM before load (approx)
    qwen_result['vram_before'] = vram_snapshot()
    # Instantiate model (device selection inside might occur)
    start_model = time.time()
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    tts = Qwen3TTS('Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice', device=device)
    model_load_time = time.time() - start_model
    qwen_result['model_load_time'] = model_load_time
    qwen_result['vram_after_load'] = vram_snapshot()
    # Generate audio
    start_gen = time.time()
    audio_data = tts.synthesize(text=text, instruction=instruction, sampling_rate=12000)
    gen_time = time.time() - start_gen
    qwen_result['gen_time'] = gen_time
    # Save WAV ? adapter saved with TTS.save_wav in adapter; replicate here if available
    try:
        tts.save_wav(audio_data, str(qwen_path), sampling_rate=12000)
    except Exception:
        # Fallback: if audio_data is raw numpy array, use soundfile to write
        import soundfile as sf
        sf.write(str(qwen_path), audio_data, 12000)
    qwen_result['vram_after_gen'] = vram_snapshot()
    # Validate file
    ok, info = file_ok(qwen_path)
    qwen_result['file_ok'] = ok
    qwen_result['ffprobe'] = info
    qwen_result['status'] = 'REAL_PASS' if ok else 'REAL_FAIL'
    print('QWEN3 REAL = PASS' if ok else 'QWEN3 REAL = FAIL')
except Exception as e:
    import traceback
    tb = traceback.format_exc()
    qwen_result['status'] = 'REAL_FAIL'
    qwen_result['error'] = str(e)
    qwen_result['traceback'] = tb
    print('QWEN3 REAL = FAIL')
    print(tb)


## Chatterbox-Turbo test (350M) ? REAL only
If this fails to import/instantiate, report `CHATTERBOX REAL = FAIL`.


In [ ]:
chatter_result = {'model': 'Chatterbox-Turbo-350M'}
try:
    import time
    start_load = time.time()
    from chatterbox import ChatterboxTurbo
    import torch
    load_time = time.time() - start_load
    chatter_result['load_time'] = load_time
    chatter_result['vram_before'] = vram_snapshot()
    start_model = time.time()
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    tts2 = ChatterboxTurbo(model_id='ResembleAI/chatterbox-turbo', device=device)
    model_load_time = time.time() - start_model
    chatter_result['model_load_time'] = model_load_time
    chatter_result['vram_after_load'] = vram_snapshot()
    start_gen = time.time()
    audio_data, sample_rate = tts2.synthesize(text=text, voice='default')
    gen_time = time.time() - start_gen
    chatter_result['gen_time'] = gen_time
    # Save using soundfile if available
    try:
        import soundfile as sf
        sf.write(str(chatter_path), audio_data, sample_rate)
    except Exception:
        # fallback write bytes if necessary
        with open(chatter_path, 'wb') as f:
            f.write(b'BINARY_AUDIO_PLACEHOLDER')
    chatter_result['vram_after_gen'] = vram_snapshot()
    ok, info = file_ok(chatter_path)
    chatter_result['file_ok'] = ok
    chatter_result['ffprobe'] = info
    chatter_result['status'] = 'REAL_PASS' if ok else 'REAL_FAIL'
    print('CHATTERBOX REAL = PASS' if ok else 'CHATTERBOX REAL = FAIL')
except Exception as e:
    import traceback
    tb = traceback.format_exc()
    chatter_result['status'] = 'REAL_FAIL'
    chatter_result['error'] = str(e)
    chatter_result['traceback'] = tb
    print('CHATTERBOX REAL = FAIL')
    print(tb)


In [ ]:
# Build report JSON and display audio players if files exist
report = { 'qwen': qwen_result, 'chatterbox': chatter_result, 'gpu': {} }
import torch
if torch.cuda.is_available():
    report['gpu'] = {
        'name': torch.cuda.get_device_name(0),
        'device_count': torch.cuda.device_count(),
    }
# Save report
with open(report_path, 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2)
print('Report saved to', report_path)
# Display players if available
from IPython.display import Audio, display
if qwen_path.exists():
    print('Qwen3 audio:')
    display(Audio(str(qwen_path)))
if chatter_path.exists():
    print('Chatterbox audio:')
    display(Audio(str(chatter_path)))
